# What does the client's scoring rule do with a policy?

This notebook loads the bookings, adds the two columns the scoring rule needs, and scores the simplest policy there is: sell nothing extra. It runs in under a minute. Change nothing here; your team's work goes in `src/`.

In [1]:
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
import numpy as np

import contract
import data

In [4]:
bookings = data.read_raw()
bookings["arrival_date"] = data.arrival_date(bookings)
bookings["nights"] = bookings["stays_in_weekend_nights"] + bookings["stays_in_week_nights"]
print(f"{len(bookings):,} bookings, arrivals {bookings.arrival_date.min().date()} to {bookings.arrival_date.max().date()}")

119,390 bookings, arrivals 2015-07-01 to 2017-08-31


In [3]:
train, choose, proof = contract.split_periods(bookings)
for name, part in [("train", train), ("choose", choose), ("proof", proof)]:
    print(f"{name:>6}: {len(part):>7,} bookings")

 train:  49,157 bookings
choose:  29,546 bookings
 proof:  40,687 bookings


## What does selling nothing extra cost on the choose period?

Every cancellation leaves a room empty and nobody is walked. This is a floor, not the client's rule: the client already oversells with a flat rate, and your savings are measured against that.

In [4]:
nights = contract.nightly_table(choose, prob=np.zeros(len(choose)))
nights.head()

book  cancelled  expected  spread  stay_value
hotel      arrival_date                                               
City Hotel 2016-07-01     141         68       0.0     0.0  319.747589
           2016-07-02      80         33       0.0     0.0  397.106875
           2016-07-03      85         21       0.0     0.0  478.769529
           2016-07-04      92         30       0.0     0.0  440.365543
           2016-07-05      74         14       0.0     0.0  405.705811

In [5]:
contract.night_cost(nights, oversell=np.zeros(len(nights)))

{'cost_eur': 4155340.86,
 'empty_rooms': 10894,
 'walked_guests': 0,
 'walks_per_1000': 0.0,
 'ceiling_met': True,
 'nights': 368}

### Checking current cancellation rate

In [5]:
print("Bookings:", len(bookings))
print("Columns:", len(bookings.columns))
print("No of Hotels:", bookings["hotel"].nunique())

Bookings: 119390
Columns: 34
Hotels: 2


In [6]:
cancellations = bookings["is_canceled"].sum()
rate = bookings["is_canceled"].mean()
print("No of cancellations ", cancellations)
print("Current hotel cancellation rate:", rate)

No of cancellations  44224
Current hotel cancellation rate: 0.37041628277075134


### Check cancellations by hotel

In [7]:
bookings.groupby("hotel")["is_canceled"].agg(
    bookings="count",
    cancelled="sum",
    cancellation_rate="mean"
)

,bookings,cancelled,cancellation_rate
hotel,,,
City Hotel,79330,33102,0.417270
Resort Hotel,40060,11122,0.277634


### Non Refund bookings

In [8]:
non_refund = bookings[bookings["deposit_type"] == "Non Refund"]

In [9]:
non_refund

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date,arrival_date,nights
2133,Resort Hotel,1,87,2015,October,40,3,0,1,2,...,NaN,0,Transient,36.05,0,0,Canceled,2015-08-12,2015-10-03,1
2134,Resort Hotel,1,87,2015,October,40,3,0,1,2,...,NaN,0,Transient,36.05,0,0,Canceled,2015-08-12,2015-10-03,1
2135,Resort Hotel,1,87,2015,October,40,3,0,1,2,...,NaN,0,Transient,36.05,0,0,Canceled,2015-08-12,2015-10-03,1
2136,Resort Hotel,1,87,2015,October,40,3,0,1,2,...,NaN,0,Transient,36.05,0,0,Canceled,2015-08-12,2015-10-03,1
2141,Resort Hotel,1,87,2015,October,40,3,0,1,2,...,NaN,0,Transient,36.05,0,0,Canceled,2015-08-12,2015-10-03,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95424,City Hotel,0,172,2016,August,34,16,0,3,1,...,51.0,0,Transient-Party,106.00,0,0,Check-Out,2016-08-19,2016-08-16,3
95936,City Hotel,0,178,2016,August,35,22,1,3,1,...,51.0,0,Transient-Party,106.00,0,0,Check-Out,2016-08-26,2016-08-22,4
96193,City Hotel,0,198,2016,August,35,25,2,3,2,...,NaN,0,Transient-Party,72.25,0,0,Check-Out,2016-08-30,2016-08-25,5
96232,City Hotel,0,186,2016,August,35,26,2,2,2,...,NaN,0,Transient-Party,83.83,0,0,Check-Out,2016-08-30,2016-08-26,4


#### Non Refund by Market Segment:

In [12]:
print(non_refund["market_segment"].value_counts(normalize=True))


Non Refund by Market Segment:
market_segment
Groups           0.628779
Offline TA/TO    0.343182
Corporate        0.022897
Online TA        0.003839
Direct           0.001303
Name: proportion, dtype: float64


Non Refund by Distribution Channel:

In [13]:

print(non_refund["distribution_channel"].value_counts(normalize=True))


Non Refund by Distribution Channel:
distribution_channel
TA/TO        0.935833
Corporate    0.036882
Direct       0.027285
Name: proportion, dtype: float64


## What comes next?

- Check every number in `BRIEF.md` against `bookings`, and record each one in the charter.
- Formalise the client's flat rule in `src/backtest.py` and score it on these same nights.
- Leave the proof period alone until every setting is frozen.